# V2 — Binoculars: generacja + ewaluacja (Kaggle)

Pełny pipeline `binoculars` w **dwóch subprocessach** (gen → eval).

Kaggle ma **więcej RAM systemowego** niż Colab — łatwiej załadować Falcon z HF.
GPU: zwykle P100/T4 ~16 GB — Falcon 4-bit/8-bit sekwencyjnie.

Pliki projektu: `/kaggle/input/datasets/.../sources/` (wgraj V2).
Secret: `HF_TOKEN`.

Po sesji pobierz `runs_backup_*.zip` z zakładki **Output**.

In [ ]:
import os
import shutil
from pathlib import Path

INPUT_DIR = Path('/kaggle/input/datasets/kamilml/sources1')
PROJECT_DIR = Path('/kaggle/working/Steganography_benchmarks_V2')

V2_FILES = [
    'generate_responses.py',
    'evaluate_responses.py',
    'eval_handlers.py',
    'common.py',
    'code_extract.py',
    'prompts.py',
    'model_runtime.py',
    'raw_store.py',
    'dummy_processor.py',
    'humaneval_subset_eval.py',
    'perplexity_metrics.py',
    'binoculars_scorer.py',
    'notebook_runner.py',
    'requirements.txt',
]

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
missing = [name for name in V2_FILES if not (INPUT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Brakuje w {INPUT_DIR}: {missing}')

for name in V2_FILES:
    shutil.copy2(INPUT_DIR / name, PROJECT_DIR / name)
    print(f'copied {name}')

os.chdir(PROJECT_DIR)
print('cwd:', Path.cwd())

In [ ]:
!pip install -q -r requirements.txt
!pip install -q "bitsandbytes>=0.43.1" "accelerate>=0.33.0"

import bitsandbytes  # noqa: F401
print('bitsandbytes OK')

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

if not os.getenv('HF_TOKEN'):
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

In [ ]:
# --- KONFIGURACJA ---
MODELS = ['qwen', 'llama', 'gemma']
THRESHOLDS = [0.0, 0.01, 0.05, 0.1]

TOP_N = 16
MAX_NEW_TOKENS = 512
SKIP_IF_EVALUATED = True
CONTINUE_ON_ERROR = True

import gc
import json
import shutil
import sys
from datetime import datetime
from pathlib import Path

import torch
from notebook_runner import run_live

TEST = 'binoculars'
PLATFORM = 'kaggle'
RUNS_DIR = Path('runs')


def free_gpu() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()


def backup_runs() -> None:
    runs = Path('runs')
    if not runs.exists() or not list(runs.rglob('*.jsonl')):
        return
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    archive = shutil.make_archive(f'/kaggle/working/runs_backup_{ts}', 'zip', runs)
    print(f'Backup: {archive}')


def _th_match(a: float, b: float) -> bool:
    return abs(float(a) - float(b)) < 1e-9


def find_run(runs_dir: Path, model: str, threshold: float) -> Path | None:
    if not runs_dir.exists():
        return None
    candidates: list[tuple[str, Path]] = []
    for manifest_path in runs_dir.glob('*/manifest.json'):
        try:
            m = json.loads(manifest_path.read_text(encoding='utf-8'))
        except json.JSONDecodeError:
            continue
        if m.get('test') != TEST:
            continue
        if m.get('model_key') != model:
            continue
        if not _th_match(m.get('threshold', -1), threshold):
            continue
        if m.get('status') != 'completed':
            continue
        candidates.append((m.get('updated_at', ''), manifest_path.parent))
    if not candidates:
        return None
    candidates.sort(key=lambda x: x[0])
    return candidates[-1][1]


def has_eval(run_dir: Path) -> bool:
    return (run_dir / 'evaluation' / 'binoculars_results.json').is_file()


def build_gen_cmd(model: str, threshold: float, run_dir: Path | None = None) -> list[str]:
    cmd = [
        sys.executable, 'generate_responses.py',
        '--test', TEST,
        '--threshold', str(threshold),
        '--top-n', str(TOP_N),
        '--max-new-tokens', str(MAX_NEW_TOKENS),
        '--platform', PLATFORM,
        '--output-root', 'runs',
    ]
    if run_dir is not None:
        cmd += ['--run-dir', str(run_dir)]
    else:
        cmd += ['--model', model]
    return cmd


def build_eval_cmd(run_dir: Path) -> list[str]:
    return [
        sys.executable, 'evaluate_responses.py',
        '--run-dir', str(run_dir),
        '--platform', PLATFORM,
    ]


jobs = [(m, th) for m in MODELS for th in THRESHOLDS]
print(f'Plan: {len(jobs)} runów binoculars (Kaggle)')

ok, skipped, failed = [], [], []

try:
    for index, (model, threshold) in enumerate(jobs, start=1):
        label = f'{model} | th={threshold}'
        print('\n' + '=' * 60)
        print(f'[{index}/{len(jobs)}] {label}')
        print('=' * 60)

        free_gpu()
        run_dir = find_run(RUNS_DIR, model, threshold)

        if SKIP_IF_EVALUATED and run_dir is not None and has_eval(run_dir):
            print(f'Pomijam — już ewaluowany: {run_dir.name}')
            skipped.append(label)
            continue

        try:
            if run_dir is None:
                print('[1/2] Generacja...')
                run_live(build_gen_cmd(model, threshold))
                run_dir = find_run(RUNS_DIR, model, threshold)
                if run_dir is None:
                    raise RuntimeError('Brak run dir po generacji')
            else:
                print(f'[1/2] RAW OK: {run_dir.name}')

            free_gpu()
            print('[2/2] Binoculars eval...')
            run_live(build_eval_cmd(run_dir))

            if not has_eval(run_dir):
                raise RuntimeError('Brak binoculars_results.json')

            data = json.loads((run_dir / 'evaluation' / 'binoculars_results.json').read_text())
            ok.append(label)
            print(
                f"OK: baseline={data['baseline_binoculars_score']:.3f}, "
                f"stego={data['binoculars_score']:.3f}, "
                f"AI rate={data['ai_detection_rate']:.1%}"
            )
        except Exception as exc:
            print(f'BŁĄD: {exc}')
            failed.append((label, str(exc)))
            if not CONTINUE_ON_ERROR:
                raise
        finally:
            free_gpu()
finally:
    backup_runs()

print('\nPODSUMOWANIE')
print(f'OK: {len(ok)} | Pominięte: {len(skipped)} | Błędy: {len(failed)}')

In [ ]:
# Tabela wyników
import json
from pathlib import Path

for manifest_path in sorted(Path('runs').glob('*/manifest.json')):
    m = json.loads(manifest_path.read_text())
    if m.get('test') != 'binoculars':
        continue
    ev = manifest_path.parent / 'evaluation' / 'binoculars_results.json'
    if not ev.is_file():
        print(f"{m['model_key']} th={m['threshold']} — brak eval")
        continue
    d = json.loads(ev.read_text())
    print(
        f"{m['model_key']:<6} th={m['threshold']:<4} "
        f"baseline={d['baseline_binoculars_score']:.3f} "
        f"stego={d['binoculars_score']:.3f} "
        f"AI={d['ai_detection_rate']:.1%}"
    )